# K-Vecinos Más Cercanos (KNN) con datos artificiales generados en scikit-learn

**Objetivo.** Generar un conjunto de datos artificial con `sklearn`, dividirlo en entrenamiento y prueba, e implementar el algoritmo KNN **desde cero** siguiendo exactamente el pseudocódigo visto en clase. Al final se compara contra `KNeighborsClassifier` de scikit-learn para validar que la implementación es correcta.

**Pseudocódigo (pizarrón):**

- **Input:** $X$, $Y$, $k$, $\hat{x}$
- **Output:** $\hat{y}$
- **Inicio**
  1. Para todo $x_i \in X$, calcula la distancia $d(\hat{x}, x_i)$
  2. Ordena de menor a mayor las distancias $d$
  3. Selecciona las $k$ menores distancias
  4. Con voto mayoritario calcula $\hat{y}$

**Distancia euclidiana (2 dimensiones):** $d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$

> Este notebook funciona tal cual en Google Colab: solo usa librerías que Colab ya trae instaladas (`numpy`, `matplotlib`, `scikit-learn`). Ejecute las celdas en orden con *Entorno de ejecución → Ejecutar todas*.

## 1. Importación de librerías

In [ ]:
# ------------------------------------------------------------
# Se importan únicamente librerías preinstaladas en Google Colab,
# por lo que no es necesario ejecutar ningún 'pip install'.
# ------------------------------------------------------------
import numpy as np                      # Operaciones numéricas y vectorizadas
import matplotlib.pyplot as plt         # Gráficas
from collections import Counter         # Conteo de votos (paso 4 del algoritmo)

from sklearn.datasets import make_blobs                 # Generador de datos artificiales
from sklearn.model_selection import train_test_split    # División entrenamiento / prueba
from sklearn.neighbors import KNeighborsClassifier      # KNN de sklearn (solo para comparar)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Se fija una semilla para que los resultados sean REPRODUCIBLES:
# cada vez que se ejecute el notebook se obtendrán los mismos datos.
SEMILLA = 42
np.random.seed(SEMILLA)

## 2. Generación del conjunto de datos artificial

Se utiliza `make_blobs`, que crea "nubes" de puntos alrededor de centros. Se eligieron **3 clases** y **2 características** ($x_1$, $x_2$) para reproducir el ejemplo del pizarrón (triángulos, círculos y cuadrados) y poder graficarlo en el plano.

In [ ]:
# ------------------------------------------------------------
# Parámetros del conjunto de datos (puede modificarlos para experimentar)
# ------------------------------------------------------------
N_MUESTRAS   = 300   # Total de puntos generados
N_CLASES     = 3     # Número de clases (triángulo, círculo, cuadrado)
DESV_ESTANDAR = 1.5  # Dispersión de cada nube; mayor valor = clases más mezcladas

# Se definen los centros manualmente y relativamente cercanos entre sí para que
# las clases se TRASLAPEN un poco (como en el pizarrón, donde círculos y cuadrados
# quedan juntos). Si se dejaran centros aleatorios muy separados, cualquier
# clasificador acertaría el 100 % y el ejercicio no sería ilustrativo.
CENTROS = [[0.0, 4.0],    # Clase 0 -> Triángulo (arriba)
           [-2.5, 0.0],   # Clase 1 -> Círculo (abajo a la izquierda)
           [2.5, 0.0]]    # Clase 2 -> Cuadrado (abajo a la derecha)

# make_blobs devuelve:
#   X -> matriz de forma (N_MUESTRAS, 2) con las coordenadas (x1, x2)
#   Y -> vector de forma (N_MUESTRAS,) con la etiqueta de clase (0, 1 o 2)
X, Y = make_blobs(
    n_samples=N_MUESTRAS,
    n_features=2,            # 2 características para poder graficar en 2D
    centers=CENTROS,         # Centros definidos arriba (uno por clase)
    cluster_std=DESV_ESTANDAR,
    random_state=SEMILLA
)

# Nombres legibles de cada clase, igual que los símbolos del pizarrón
NOMBRES_CLASES = {0: "Triángulo", 1: "Círculo", 2: "Cuadrado"}
MARCADORES     = {0: "^",         1: "o",       2: "s"}
COLORES        = {0: "tab:blue",  1: "tab:orange", 2: "tab:green"}

assert len(CENTROS) == N_CLASES, "Debe haber un centro por clase"
print("Forma de X:", X.shape)
print("Forma de Y:", Y.shape)
print("Muestras por clase:", {NOMBRES_CLASES[c]: int(n) for c, n in zip(*np.unique(Y, return_counts=True))})
print("\nPrimeras 5 filas de X:\n", X[:5])
print("Primeras 5 etiquetas de Y:", Y[:5])

## 3. División en conjunto de entrenamiento y conjunto de prueba

- **Entrenamiento (70 %)**: son los puntos "conocidos" ($X$, $Y$) contra los que se medirán las distancias.
- **Prueba (30 %)**: puntos que el modelo **no ha visto**; cada uno funciona como un $\hat{x}$ nuevo cuya clase $\hat{y}$ hay que predecir.

Se usa `stratify=Y` para que cada clase conserve la misma proporción en ambos conjuntos.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.30,        # 30 % de los datos se reservan para prueba
    random_state=SEMILLA,  # División reproducible
    stratify=Y             # Mantiene la proporción de clases
)

print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba:        {X_test.shape[0]} muestras")

## 4. Visualización del conjunto de datos

In [ ]:
# ------------------------------------------------------------
# Se grafican los puntos de entrenamiento (rellenos) y los de prueba
# (contorno negro) para distinguir ambos conjuntos.
# ------------------------------------------------------------
plt.figure(figsize=(8, 6))
for c in range(N_CLASES):
    plt.scatter(X_train[Y_train == c, 0], X_train[Y_train == c, 1],
                marker=MARCADORES[c], c=COLORES[c], alpha=0.6,
                label=f"{NOMBRES_CLASES[c]} (entrenamiento)")
    plt.scatter(X_test[Y_test == c, 0], X_test[Y_test == c, 1],
                marker=MARCADORES[c], c=COLORES[c], edgecolors="black", s=70,
                label=f"{NOMBRES_CLASES[c]} (prueba)")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Conjunto de datos artificial generado con make_blobs")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

## 5. Implementación de KNN desde cero (siguiendo el pizarrón)

Cada paso del pseudocódigo está marcado con un comentario `# PASO n`.

In [ ]:
def distancia_euclidiana(a, b):
    """
    Calcula la distancia euclidiana entre dos puntos a y b.
    Fórmula del pizarrón para 2D: d = sqrt((x1 - x2)^2 + (y1 - y2)^2).
    Se escribe de forma general (suma de cuadrados de las diferencias),
    así funciona con cualquier número de características.
    """
    return np.sqrt(np.sum((a - b) ** 2))


def knn_predecir_punto(X, Y, k, x_gorro):
    """
    Predice la clase de UN punto nuevo.

    Input:
        X       -> puntos de entrenamiento, forma (n, m)
        Y       -> etiquetas de entrenamiento, forma (n,)
        k       -> número de vecinos a considerar
        x_gorro -> punto nuevo a clasificar (x̂), forma (m,)
    Output:
        y_gorro -> clase predicha (ŷ)
    """
    # PASO 1: Para todo x_i en X, calcula la distancia d(x̂, x_i).
    #         Se guarda cada distancia junto con la etiqueta de su punto.
    distancias = []
    for x_i, y_i in zip(X, Y):
        d = distancia_euclidiana(x_gorro, x_i)
        distancias.append((d, y_i))

    # PASO 2: Ordena de menor a mayor las distancias.
    #         key=lambda t: t[0] indica que se ordene por la distancia.
    distancias.sort(key=lambda t: t[0])

    # PASO 3: Selecciona las k menores distancias (los k vecinos más cercanos).
    k_vecinos = distancias[:k]
    etiquetas_vecinos = [etiqueta for _, etiqueta in k_vecinos]

    # PASO 4: Con voto mayoritario calcula ŷ.
    #         Counter cuenta cuántas veces aparece cada clase entre los vecinos;
    #         most_common(1) devuelve la clase con más votos.
    #         En caso de empate, Counter conserva la clase que apareció primero,
    #         es decir, la del vecino más cercano (criterio de desempate razonable).
    y_gorro = Counter(etiquetas_vecinos).most_common(1)[0][0]
    return y_gorro


def knn_predecir(X, Y, k, X_nuevos):
    """Aplica knn_predecir_punto a cada punto de un conjunto de prueba."""
    return np.array([knn_predecir_punto(X, Y, k, x) for x in X_nuevos])

## 6. Ejemplo paso a paso con un solo punto $\hat{x}$

Se toma un punto del conjunto de prueba y se muestra qué ocurre en cada paso, igual que en el dibujo del pizarrón (líneas rojas del punto nuevo hacia sus vecinos).

In [ ]:
k = 5                 # Número de vecinos (se recomienda impar para reducir empates)
# Índice del punto de prueba que se usará como x̂. Se eligió el 1 porque sus
# 5 vecinos coinciden claramente. Pruebe cambiarlo a 0: ahí los vecinos quedan
# 2-2-1 (empate) y se predice la clase del vecino más cercano; ese punto está
# en la zona donde las tres clases se mezclan.
INDICE_EJEMPLO = 1
x_gorro = X_test[INDICE_EJEMPLO]   # Punto nuevo a clasificar
y_real  = Y_test[INDICE_EJEMPLO]   # Clase verdadera (solo para verificar)

# Se repiten los pasos 1-3 de forma vectorizada para poder mostrarlos
d_todas = np.sqrt(np.sum((X_train - x_gorro) ** 2, axis=1))   # PASO 1
indices_ordenados = np.argsort(d_todas)                         # PASO 2
indices_k = indices_ordenados[:k]                               # PASO 3

print(f"Punto x̂ = {np.round(x_gorro, 3)}  |  clase real: {NOMBRES_CLASES[y_real]}\n")
print(f"Los {k} vecinos más cercanos:")
for pos, idx in enumerate(indices_k, start=1):
    print(f"  {pos}. distancia = {d_todas[idx]:.3f}  ->  {NOMBRES_CLASES[Y_train[idx]]}")

y_gorro = knn_predecir_punto(X_train, Y_train, k, x_gorro)      # PASO 4
print(f"\nVoto mayoritario: ŷ = {NOMBRES_CLASES[y_gorro]}")
print("¿Predicción correcta?", "Sí" if y_gorro == y_real else "No")

# Gráfica estilo pizarrón: líneas rojas desde x̂ hacia sus k vecinos
plt.figure(figsize=(8, 6))
for c in range(N_CLASES):
    plt.scatter(X_train[Y_train == c, 0], X_train[Y_train == c, 1],
                marker=MARCADORES[c], c=COLORES[c], alpha=0.4, label=NOMBRES_CLASES[c])
for idx in indices_k:
    plt.plot([x_gorro[0], X_train[idx, 0]], [x_gorro[1], X_train[idx, 1]], "r-", lw=1)
plt.scatter(*x_gorro, marker="X", c="red", s=200, edgecolors="black", label="x̂ (punto nuevo)")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.title(f"KNN con k = {k}: x̂ y sus vecinos más cercanos")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 7. Evaluación sobre todo el conjunto de prueba

In [ ]:
# Se predice la clase de cada punto de prueba con la implementación propia
Y_pred_propio = knn_predecir(X_train, Y_train, k, X_test)

exactitud = accuracy_score(Y_test, Y_pred_propio)
print(f"Exactitud (accuracy) de KNN propio con k = {k}: {exactitud:.4f}\n")

print("Matriz de confusión (filas = clase real, columnas = clase predicha):")
print(confusion_matrix(Y_test, Y_pred_propio))

print("\nReporte de clasificación:")
print(classification_report(Y_test, Y_pred_propio,
                            target_names=[NOMBRES_CLASES[c] for c in range(N_CLASES)]))

## 8. Validación contra `KNeighborsClassifier` de scikit-learn

Para comprobar que la implementación es correcta, se entrena el KNN de sklearn con los mismos parámetros (distancia euclidiana, voto uniforme) y se comparan las predicciones.

In [ ]:
knn_sklearn = KNeighborsClassifier(
    n_neighbors=k,
    metric="euclidean",   # Misma distancia que la implementación propia
    weights="uniform"     # Cada vecino vale un voto (voto mayoritario simple)
)
knn_sklearn.fit(X_train, Y_train)
Y_pred_sklearn = knn_sklearn.predict(X_test)

print(f"Exactitud KNN propio:  {accuracy_score(Y_test, Y_pred_propio):.4f}")
print(f"Exactitud KNN sklearn: {accuracy_score(Y_test, Y_pred_sklearn):.4f}")

coincidencias = np.mean(Y_pred_propio == Y_pred_sklearn) * 100
print(f"\nLas predicciones coinciden en el {coincidencias:.1f} % de los puntos de prueba.")
# Nota: si hubiera alguna diferencia mínima, se debería a la forma de desempatar
# votos o distancias idénticas; con k impar y datos continuos es muy poco probable.

## 9. Efecto del valor de $k$

Se prueba la implementación propia con distintos valores de $k$ para observar cómo cambia la exactitud. Un $k$ muy pequeño es sensible al ruido; uno muy grande suaviza demasiado la frontera entre clases.

In [ ]:
valores_k = list(range(1, 32, 2))   # Solo valores impares: 1, 3, 5, ..., 31
exactitudes = [accuracy_score(Y_test, knn_predecir(X_train, Y_train, kk, X_test))
               for kk in valores_k]

mejor_k = valores_k[int(np.argmax(exactitudes))]
print(f"Mejor k encontrado: {mejor_k}  (exactitud = {max(exactitudes):.4f})")

plt.figure(figsize=(8, 4))
plt.plot(valores_k, exactitudes, "o-")
plt.xlabel("k (número de vecinos)")
plt.ylabel("Exactitud en prueba")
plt.title("Exactitud de KNN según el valor de k")
plt.xticks(valores_k)
plt.grid(alpha=0.3)
plt.show()

## 10. Frontera de decisión

Se clasifica cada punto de una malla que cubre el plano; el color de fondo indica la clase que KNN asignaría a un $\hat{x}$ ubicado en esa posición.

In [ ]:
# Se usa el KNN de sklearn en esta celda solo por velocidad (la malla tiene miles
# de puntos); ya se comprobó en la sección 8 que da los mismos resultados.
paso = 0.1
x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x1_min, x1_max, paso), np.arange(x2_min, x2_max, paso))

Z = knn_sklearn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
# Se usa un mapa de colores con los MISMOS colores de cada clase, para que el
# fondo de cada región coincida con el color de sus puntos.
from matplotlib.colors import ListedColormap
mapa_colores = ListedColormap([COLORES[c] for c in range(N_CLASES)])
plt.contourf(xx, yy, Z, alpha=0.2, cmap=mapa_colores, levels=np.arange(-0.5, N_CLASES))
for c in range(N_CLASES):
    plt.scatter(X_test[Y_test == c, 0], X_test[Y_test == c, 1],
                marker=MARCADORES[c], c=COLORES[c], edgecolors="black", s=60,
                label=f"{NOMBRES_CLASES[c]} (prueba)")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.title(f"Frontera de decisión de KNN (k = {k})")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 11. Conclusiones

- Se generó un conjunto de datos artificial de 300 puntos, 2 características y 3 clases con `make_blobs` de scikit-learn.
- Se implementó KNN desde cero siguiendo los cuatro pasos del pseudocódigo: cálculo de distancias, ordenamiento, selección de los $k$ menores y voto mayoritario.
- La implementación propia obtiene las mismas predicciones que `KNeighborsClassifier`, lo que valida su correcto funcionamiento.
- El valor de $k$ influye en la exactitud; conviene elegirlo probando varios valores, como en la sección 9.

**Para experimentar:** modifique `DESV_ESTANDAR` (sección 2) a un valor mayor, por ejemplo 3.0, para que las clases se traslapen y observe cómo cambia el mejor $k$.